# Unidad 2 — Probabilidad

En esta unidad pasamos de **describir datos** a **modelar incertidumbre**. Las preguntas
guía son:

- ¿Cómo combinamos eventos? (uniones, intersecciones, complementos)
- ¿Qué cambia cuando _ya sabemos_ algo? (probabilidad condicional)
- ¿Cómo le damos vuelta a una pregunta? (Bayes)

Las fórmulas viven en `src/symbolic/theorems.py`; los cálculos numéricos en
`src/probability/`. Las visualizaciones interactivas en `src/widgets/bayes_explorer.py`.


## Importaciones

In [ ]:
from core import Settings
from exercises import NumericAnswerInput, verify_numeric_answer
from probability import (
    BayesInput,
    ConditionalInput,
    JointEventInput,
    SetOperationInput,
    evaluate_bayes,
    evaluate_conditional_probability,
    evaluate_set_operations,
    joint_event_probabilities,
)
from probability.total_probability import TotalProbabilityBranch
from symbolic import bayes_theorem, total_probability_theorem
from widgets import BayesExplorerInput, build_bayes_explorer

In [ ]:
settings = Settings()

## Concreto: un experimento con dado

Tiramos un dado equilibrado. Definimos:

- $A$ = "sale par" = $\{2, 4, 6\}$.
- $B$ = "sale mayor o igual que 4" = $\{4, 5, 6\}$.

Las operaciones de conjuntos son la **maquinaria mecánica** detrás de toda la unidad.


In [ ]:
universe = frozenset({"1", "2", "3", "4", "5", "6"})
event_even = frozenset({"2", "4", "6"})
event_at_least_four = frozenset({"4", "5", "6"})

set_result = evaluate_set_operations(
    SetOperationInput(
        universe=universe,
        event_a=event_even,
        event_b=event_at_least_four,
    )
)
set_result

### Pictórico: diagrama de Venn mental

Imaginá dos círculos solapados: $A \cap B = \{4, 6\}$. La unión $A \cup B$ es todo lo
que cae en alguno de los dos. La **intuición Singapur** es esta:

$$ |A \cup B| = |A| + |B| - |A \cap B| $$

Si los sumás directamente, **contás dos veces** la intersección. Por eso se la resta.


## Abstracto: la regla aditiva en términos de probabilidad

Dividiendo cardinalidades por $|\Omega| = 6$ obtenemos la **regla aditiva**:

$$ P(A \cup B) = P(A) + P(B) - P(A \cap B) $$


In [ ]:
joint = joint_event_probabilities(
    JointEventInput(
        probability_a=3 / 6,
        probability_b=3 / 6,
        probability_intersection=2 / 6,
    )
)
joint

## Probabilidad condicional

$$ P(A \mid B) = \frac{P(A \cap B)}{P(B)} $$

**Intuición Singapur:** _restringir el universo_ a $B$. Dentro de ese universo nuevo
preguntamos qué fracción cae también en $A$.


In [ ]:
conditional = evaluate_conditional_probability(
    ConditionalInput(
        probability_intersection=2 / 6,
        probability_conditioning_event=3 / 6,
    )
)
conditional

## Teorema de Bayes (forma simbólica)

Antes de calcular, miramos la fórmula simbólica viviendo en `src/symbolic/theorems.py`.
Esta es la única vez que escribimos la fórmula a mano: el resto de la unidad la _reutiliza_.


In [ ]:
bayes_theorem().formula

## Bayes con datos: prueba diagnóstica

**Concreto.** Una enfermedad tiene prevalencia $P(D) = 1\%$. El test tiene sensibilidad
$99\%$ y especificidad $95\%$. Si el test da positivo, ¿cuál es $P(D \mid +)$?

**Intuición Singapur (pictórica).** Pensá en 10.000 personas. 100 tienen la enfermedad
y 99 de ellas dan positivo. De las 9.900 sanas, 495 dan positivo de todas formas
(falsos positivos). En total hay $99 + 495 = 594$ positivos; el porcentaje de "verdaderos
enfermos dentro de los positivos" es $99 / 594 \approx 16{,}7\%$.

**Abstracto:**

$$ P(D \mid +) = \frac{P(+ \mid D)\,P(D)}{P(+ \mid D)\,P(D) + P(+ \mid \bar{D})\,P(\bar{D})} $$


In [ ]:
branches = (
    TotalProbabilityBranch(label="Enfermo", prior=0.01, likelihood=0.99),
    TotalProbabilityBranch(label="Sano", prior=0.99, likelihood=0.05),
)
posteriors = evaluate_bayes(BayesInput(branches=branches))
for _posterior in posteriors:
    pass

### Por qué la intuición duele

Aunque el test es buenísimo (sensibilidad 99%), un positivo apenas mueve la creencia del
$1\%$ al $16{,}7\%$. La culpa es de la **tasa base**: como casi nadie está enfermo, los
falsos positivos pesan mucho. Movés los sliders y ves cómo la posterior reacciona.


In [ ]:
build_bayes_explorer(BayesExplorerInput(settings=settings))

## Probabilidad total (forma simbólica)

Si $\{A_1, \dots, A_k\}$ es una partición del universo:

$$ P(B) = \sum_{i=1}^{k} P(B \mid A_i)\,P(A_i) $$


In [ ]:
total_probability_theorem(partition_size=3).formula

## Ejercicio 1 — Regla aditiva

Sea $P(A) = 0{,}6$, $P(B) = 0{,}5$, $P(A \cap B) = 0{,}2$. Calcular $P(A \cup B)$.

**Idea.** Aplicar $P(A \cup B) = P(A) + P(B) - P(A \cap B) = 0{,}6 + 0{,}5 - 0{,}2 = 0{,}9$.


In [ ]:
expected_union = joint_event_probabilities(
    JointEventInput(probability_a=0.6, probability_b=0.5, probability_intersection=0.2)
).union

student_answer = 0.9
verify_numeric_answer(NumericAnswerInput(student_answer=student_answer, expected_answer=expected_union))

## Ejercicio 2 — Bayes "a mano"

Una caja $C_1$ tiene 3 bolas rojas y 7 blancas; la caja $C_2$ tiene 6 rojas y 4 blancas.
Se elige una caja al azar (probabilidad $1/2$ cada una) y se saca una bola. Resulta roja.
¿Cuál es $P(C_1 \mid \text{roja})$?

**Idea.**

$$ P(C_1 \mid R) = \frac{P(R \mid C_1)\,P(C_1)}{P(R \mid C_1)\,P(C_1) + P(R \mid C_2)\,P(C_2)}
 = \frac{0{,}3 \cdot 0{,}5}{0{,}3 \cdot 0{,}5 + 0{,}6 \cdot 0{,}5} = \frac{0{,}15}{0{,}45} = \frac{1}{3} $$


In [ ]:
box_branches = (
    TotalProbabilityBranch(label="C1", prior=0.5, likelihood=0.3),
    TotalProbabilityBranch(label="C2", prior=0.5, likelihood=0.6),
)
expected_posterior = evaluate_bayes(BayesInput(branches=box_branches))[0].posterior

student_answer = 1 / 3
verify_numeric_answer(NumericAnswerInput(student_answer=student_answer, expected_answer=expected_posterior))

## Para llevarse

- La regla aditiva existe **para no contar dos veces** la intersección.
- Condicionar = **restringir el universo**.
- Bayes invierte el sentido del condicionamiento usando la probabilidad total como denominador.
- La tasa base manda más de lo que la intuición sugiere.
